In [ ]:
# Azure AI Language Python client library: pip install azure-ai-textanalytics
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import os, time, uuid, random, re, html

In [ ]:
load_dotenv(".env")

LANGUAGE_ENDPOINT = https://REDACTED_ENDPOINT/
LANGUAGE_KEY = REDACTED_AZURE_KEY

if not LANGUAGE_ENDPOINT or not LANGUAGE_KEY:
    raise ValueError("Missing AZURE_LANGUAGE_ENDPOINT or AZURE_LANGUAGE_KEY in .env")

client = TextAnalyticsClient(
    endpoint=LANGUAGE_ENDPOINT,
    credential=AzureKeyCredential(LANGUAGE_KEY)
)

print("Azure Language client ready.")

In [ ]:
# =========================
# SOURCE FOLDER
# =========================
SOURCE_DIR = Path(r"C:\Users\Edwin\Desktop\Lithan Academy\ModellingProject\datasets\abc_bank_reviews_text_analysis")

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Folder not found: {SOURCE_DIR}")

print("Reading files from:", SOURCE_DIR)

In [ ]:
# =========================
# HELPERS
# =========================
def strip_html_tags(text: str) -> str:
    text = html.unescape(text or "")
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"</p\s*>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_eml_body(eml_path: Path) -> str:
    with eml_path.open("rb") as f:
        msg = BytesParser(policy=policy.default).parse(f)

    plain_parts = []
    html_parts = []

    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            disposition = str(part.get_content_disposition() or "").lower()

            if disposition == "attachment":
                continue

            try:
                content = part.get_content()
            except Exception:
                continue

            if not content:
                continue

            if content_type == "text/plain":
                plain_parts.append(str(content))
            elif content_type == "text/html":
                html_parts.append(str(content))
    else:
        try:
            content = msg.get_content()
        except Exception:
            content = ""

        if msg.get_content_type() == "text/plain":
            plain_parts.append(str(content))
        elif msg.get_content_type() == "text/html":
            html_parts.append(str(content))

    if plain_parts:
        return "\n".join(plain_parts).strip()

    if html_parts:
        return strip_html_tags("\n".join(html_parts))

    return ""


def extract_text_file(file_path: Path) -> str:
    # try utf-8 first, then fallback
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin-1"]
    for enc in encodings:
        try:
            return file_path.read_text(encoding=enc).strip()
        except Exception:
            continue
    return ""


def extract_email_text(file_path: Path) -> str:
    ext = file_path.suffix.lower()

    if ext == ".eml":
        return extract_eml_body(file_path)

    if ext in [".txt", ".log"]:
        return extract_text_file(file_path)

    return ""


def analyze_email_text(text: str):
    result = client.analyze_sentiment(
        [text],
        show_opinion_mining=True
    )[0]

    if result.is_error:
        raise RuntimeError(f"Analysis failed: {result.error}")

    opinions = []
    sentence_rows = []

    for sentence in result.sentences:
        sentence_rows.append({
            "sentence_text": sentence.text,
            "sentence_sentiment": sentence.sentiment,
            "sentence_positive": sentence.confidence_scores.positive,
            "sentence_neutral": sentence.confidence_scores.neutral,
            "sentence_negative": sentence.confidence_scores.negative,
        })

        for opinion in sentence.mined_opinions:
            target_text = opinion.target.text
            target_sentiment = opinion.target.sentiment

            for assessment in opinion.assessments:
                opinions.append({
                    "target": target_text,
                    "target_sentiment": target_sentiment,
                    "assessment": assessment.text,
                    "assessment_sentiment": assessment.sentiment,
                    "is_negated": assessment.is_negated,
                    "sentence_text": sentence.text
                })

    return {
        "document_sentiment": result.sentiment,
        "document_positive": result.confidence_scores.positive,
        "document_neutral": result.confidence_scores.neutral,
        "document_negative": result.confidence_scores.negative,
        "opinions": opinions,
        "sentences": sentence_rows
    }

In [ ]:
# =========================
# PROCESS ALL EMAIL FILES
# =========================
supported_files = []
for pattern in ["*.eml", "*.txt", "*.log"]:
    supported_files.extend(SOURCE_DIR.glob(pattern))

supported_files = sorted(supported_files)

if not supported_files:
    raise FileNotFoundError("No .eml, .txt, or .log files found in the folder.")

print(f"Found {len(supported_files)} files")

results = []
opinions_output = []
sentences_output = []

for file_path in supported_files:
    print("Processing:", file_path.name)

    text = extract_email_text(file_path)

    if not text.strip():
        print(f"  Skipped - no readable text: {file_path.name}")
        continue

    try:
        analysis = analyze_email_text(text)

        results.append({
            "file_name": file_path.name,
            "file_path": str(file_path),
            "char_count": len(text),
            "document_sentiment": analysis["document_sentiment"],
            "document_positive": analysis["document_positive"],
            "document_neutral": analysis["document_neutral"],
            "document_negative": analysis["document_negative"],
            "text_preview": text[:300]
        })

        for op in analysis["opinions"]:
            opinions_output.append({
                "file_name": file_path.name,
                **op
            })

        for sent in analysis["sentences"]:
            sentences_output.append({
                "file_name": file_path.name,
                **sent
            })

    except Exception as e:
        results.append({
            "file_name": file_path.name,
            "file_path": str(file_path),
            "char_count": len(text),
            "document_sentiment": "ERROR",
            "document_positive": None,
            "document_neutral": None,
            "document_negative": None,
            "text_preview": str(e)
        })
        print(f"  Error: {e}")

df_results = pd.DataFrame(results)
df_opinions = pd.DataFrame(opinions_output)
df_sentences = pd.DataFrame(sentences_output)

print("Done.")

In [ ]:
# =========================
# VIEW RESULTS
# =========================
display(df_results)
display(df_opinions.head(20))
display(df_sentences.head(20))

In [ ]:
# =========================
# EXPORT TO CSV
# =========================
output_dir = SOURCE_DIR / "analysis_output"
output_dir.mkdir(exist_ok=True)

results_csv = output_dir / "email_sentiment_results.csv"
opinions_csv = output_dir / "email_opinion_mining.csv"
sentences_csv = output_dir / "email_sentence_sentiment.csv"

df_results.to_csv(results_csv, index=False, encoding="utf-8-sig")
df_opinions.to_csv(opinions_csv, index=False, encoding="utf-8-sig")
df_sentences.to_csv(sentences_csv, index=False, encoding="utf-8-sig")

print("Saved:")
print(results_csv)
print(opinions_csv)
print(sentences_csv)